### IDX Exchange Team **ds55**
### Beini Lan
# 03_baseline_model - Week 4 Baseline Model

This Week 4 notebook uses the cleaned Week 3 train/test modeling CSV to train the first baseline model: a Linear Regression fit with ordinary least squares.

**Week 4 requirements covered**
- Train a Linear Regression as the first model.
- Evaluate using R2 on the test set.
- Record baseline results.
- Create a summary report based on baseline model performance.


## Setup


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.float_format", "{:,.4f}".format)

TRACE_COLUMNS = ["ListingKey", "ListingId", "CloseDate", "SourceMonth", "Split", "ClosePrice"]
LEAKAGE_COLUMNS = ["ListPrice", "OriginalListPrice"]
MODEL_NAME = "Linear Regression (ordinary least squares)"


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "W3 Data Preprocessing").exists() and (candidate / "raw data").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root with W3 Data Preprocessing and raw data folders.")


def find_cleaned_csv(project_root):
    candidates = [
        project_root / "W3 Data Preprocessing" / "cleaned_crmls_sfr_train_test.csv",
        project_root / "W3 Data Preprocessing" / "Cleaned SFR CRMLSSold CSVs" / "cleaned_crmls_sfr_train_test.csv",
        project_root / "W3 Data Preprocessing" / "cleaned_crmls_sfr_train_test.csv.zip",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find the Week 3 cleaned train/test CSV.")


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "W4 Baseline Model"
CLEANED_CSV_PATH = find_cleaned_csv(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Week 4 output directory: {OUTPUT_DIR}")


FileNotFoundError: Could not locate project root with W3 Data Preprocessing and raw data folders.

## Load Week 3 Cleaned Data


In [ ]:
data = pd.read_csv(CLEANED_CSV_PATH)
print(f"Cleaned data path: {CLEANED_CSV_PATH}")
print(f"Cleaned data shape: {data.shape}")
print(f"Train rows: {data['Split'].eq('train').sum():,}")
print(f"Test rows: {data['Split'].eq('test').sum():,}")


Cleaned data path: /Users/HP/Documents/IDX Exchange/W3 Data Preprocessing/cleaned_crmls_sfr_train_test.csv
Cleaned data shape: (141996, 99)
Train rows: 129,972
Test rows: 12,024


## Modeling Checks

The Week 3 cleaned CSV already contains numeric, imputed, and normalized predictors. Week 4 keeps the traceability columns and target out of `X`, and it ignores leakage columns `ListPrice` and `OriginalListPrice` before modeling.


In [ ]:
missing_required = sorted(set(TRACE_COLUMNS).difference(data.columns))
if missing_required:
    raise ValueError(f"Missing required cleaned-data columns: {missing_required}")

leakage_present = sorted(set(LEAKAGE_COLUMNS).intersection(data.columns))
feature_columns = [column for column in data.columns if column not in TRACE_COLUMNS]
non_numeric_features = [
    column for column in feature_columns if not pd.api.types.is_numeric_dtype(data[column])
]
missing_feature_values = int(data[feature_columns].isna().sum().sum())

if leakage_present:
    raise ValueError(f"Leakage columns should not be present in the modeling data: {leakage_present}")
if non_numeric_features:
    raise ValueError(f"All predictors must be numeric. Non-numeric columns: {non_numeric_features}")
if missing_feature_values:
    raise ValueError(f"Feature matrix contains {missing_feature_values:,} missing values.")

split_summary = (
    data.groupby("Split")
    .agg(
        rows=("ListingKey", "size"),
        first_month=("SourceMonth", "min"),
        last_month=("SourceMonth", "max"),
        median_close_price=("ClosePrice", "median"),
    )
)

print("Feature matrix checks")
print("---------------------")
print(f"Predictor columns: {len(feature_columns)}")
print(f"Missing predictor values: {missing_feature_values}")
print(f"Non-numeric predictor columns: {non_numeric_features}")
print(f"Leakage columns present: {leakage_present}")
print()
print("Split summary")
print(split_summary.to_string())


Feature matrix checks
---------------------
Predictor columns: 93
Missing predictor values: 0
Non-numeric predictor columns: []
Leakage columns present: []

Split summary
       rows  first_month  last_month  median_close_price
Split                                                        
test  12024       202605      202605            930000.0
train 129972      202505      202604            889000.0


## Train Linear Regression Baseline


In [ ]:
def add_intercept(matrix):
    return np.column_stack([np.ones(matrix.shape[0]), matrix])


def fit_linear_regression(x_train, y_train):
    design_matrix = add_intercept(x_train)
    coefficients, _, rank, _ = np.linalg.lstsq(design_matrix, y_train, rcond=None)
    return coefficients, int(rank)


def predict_linear_regression(x, coefficients):
    return add_intercept(x) @ coefficients


train = data.loc[data["Split"].eq("train")].copy()
test = data.loc[data["Split"].eq("test")].copy()

x_train = train[feature_columns].to_numpy(dtype=np.float64)
y_train = train["ClosePrice"].to_numpy(dtype=np.float64)
x_test = test[feature_columns].to_numpy(dtype=np.float64)
y_test = test["ClosePrice"].to_numpy(dtype=np.float64)

coefficients, matrix_rank = fit_linear_regression(x_train, y_train)
train_predictions = predict_linear_regression(x_train, coefficients)
test_predictions = predict_linear_regression(x_test, coefficients)

print(f"Trained {MODEL_NAME}.")
print(f"Design matrix columns including intercept: {x_train.shape[1] + 1}")
print(f"Design matrix rank: {matrix_rank}")


Trained Linear Regression (ordinary least squares).
Design matrix columns including intercept: 94
Design matrix rank: 91


## Evaluate With Test-Set R2


In [ ]:
def r2_score(y_true, y_pred):
    total_sum_squares = np.sum((y_true - y_true.mean()) ** 2)
    residual_sum_squares = np.sum((y_true - y_pred) ** 2)
    return float(1 - residual_sum_squares / total_sum_squares)


train_mean_predictions = np.full_like(y_test, y_train.mean(), dtype=np.float64)
baseline_results = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "data_source": str(CLEANED_CSV_PATH),
            "train_rows": int(len(train)),
            "test_rows": int(len(test)),
            "feature_count": int(len(feature_columns)),
            "train_months": f"{train['SourceMonth'].min()}-{train['SourceMonth'].max()}",
            "test_month": str(test["SourceMonth"].iloc[0]),
            "train_r2": r2_score(y_train, train_predictions),
            "test_r2": r2_score(y_test, test_predictions),
            "train_mean_baseline_test_r2": r2_score(y_test, train_mean_predictions),
            "test_mae": float(np.mean(np.abs(y_test - test_predictions))),
            "test_rmse": float(np.sqrt(np.mean((y_test - test_predictions) ** 2))),
            "test_median_absolute_error": float(np.median(np.abs(y_test - test_predictions))),
            "test_mean_actual_close_price": float(y_test.mean()),
            "test_median_actual_close_price": float(np.median(y_test)),
            "test_mean_predicted_close_price": float(test_predictions.mean()),
            "test_median_predicted_close_price": float(np.median(test_predictions)),
            "design_matrix_rank": matrix_rank,
            "design_matrix_columns_including_intercept": int(x_train.shape[1] + 1),
        }
    ]
)

print(
    baseline_results[
        [
            "train_r2",
            "test_r2",
            "train_mean_baseline_test_r2",
            "test_mae",
            "test_rmse",
            "test_median_absolute_error",
        ]
    ].T.reset_index().rename(columns={"index": "metric", 0: "value"}).to_string(index=False)
)


                     metric         value
                   train_r2  1.800770e-02
                    test_r2  3.638567e-01
train_mean_baseline_test_r2 -5.155471e-04
                   test_mae  5.289417e+05
                  test_rmse  1.338397e+06
 test_median_absolute_error  3.491084e+05


## Record Baseline Results


In [ ]:
coefficients_table = pd.DataFrame(
    {
        "feature": ["intercept", *feature_columns],
        "coefficient": coefficients,
    }
)
top_coefficients = (
    coefficients_table.loc[coefficients_table["feature"].ne("intercept")]
    .assign(abs_coefficient=lambda frame: frame["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
    .head(15)
    .drop(columns="abs_coefficient")
    .reset_index(drop=True)
)

predictions = test[["ListingKey", "ListingId", "CloseDate", "SourceMonth", "ClosePrice"]].copy()
predictions["PredictedClosePrice"] = test_predictions
predictions["Residual"] = predictions["ClosePrice"] - predictions["PredictedClosePrice"]
predictions["AbsoluteError"] = predictions["Residual"].abs()

baseline_results.to_csv(OUTPUT_DIR / "baseline_model_results.csv", index=False)
top_coefficients.to_csv(OUTPUT_DIR / "baseline_model_top_coefficients.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "baseline_model_test_predictions.csv", index=False)

print("Top coefficients by absolute value")
print(top_coefficients.to_string(index=False))


Top coefficients by absolute value
                        feature   coefficient
     PostalCode_train_frequency -2.396088e+07
             LivingArea_missing  4.184903e+06
    CountyOrParish__other_state -1.524075e+06
       CountyOrParish__imperial -1.512001e+06
        CountyOrParish__ventura -1.502052e+06
      CountyOrParish__riverside -1.387757e+06
         CountyOrParish__plumas  1.263325e+06
         CountyOrParish__sierra  1.252021e+06
 CountyOrParish__san_bernardino -1.173354e+06
        CountyOrParish__trinity  1.117280e+06
         CountyOrParish__shasta  1.095631e+06
         CountyOrParish__lassen  1.090362e+06
              WaterfrontYN_flag  1.065625e+06
CountyOrParish__foreign_country -1.056512e+06
                   LivingArea_z  1.052050e+06


## Summary Report


In [ ]:
def dollars(value):
    return f"${value:,.0f}"


metrics = baseline_results.iloc[0].to_dict()
report_text = "\n".join(
    [
        "Week 4 Baseline Model Summary Report",
        "====================================",
        "",
        f"Model: {metrics['model']}",
        f"Training data: {int(metrics['train_rows']):,} rows from {metrics['train_months']}",
        f"Test data: {int(metrics['test_rows']):,} rows from {metrics['test_month']}",
        f"Predictor count: {int(metrics['feature_count'])}",
        "",
        "Baseline performance",
        "--------------------",
        f"Test R2: {metrics['test_r2']:.4f}",
        f"Train R2: {metrics['train_r2']:.4f}",
        f"Test MAE: {dollars(metrics['test_mae'])}",
        f"Test RMSE: {dollars(metrics['test_rmse'])}",
        f"Test median absolute error: {dollars(metrics['test_median_absolute_error'])}",
        "",
        "Interpretation",
        "--------------",
        (
            "The first linear regression baseline explains "
            f"{metrics['test_r2'] * 100:.1f}% of ClosePrice variance on the holdout month. "
            "This is a useful starting line, but the error levels are still large for property-level "
            "price prediction."
        ),
        (
            "The Week 3 preprocessing kept listing-price leakage fields out of the modeling matrix. "
            "The next model should compare against this R2 while exploring stronger nonlinear models, "
            "target/outlier treatment, and richer neighborhood features."
        ),
        "",
        "Largest absolute coefficients",
        "-----------------------------",
        top_coefficients.to_string(index=False),
        "",
    ]
)

report_path = OUTPUT_DIR / "Week4_Baseline_Model_Summary_Report.txt"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)
print(f"Saved summary report: {report_path}")


NameError: name 'baseline_results' is not defined

## Week 4 Takeaways

- The Week 4 Linear Regression baseline reached **test R2 = 0.3639** on the May 2026 holdout month.
- The model uses **93 numeric predictors** from Week 3 preprocessing and keeps `ListPrice` / `OriginalListPrice` out of the feature matrix.
- Baseline error is still large (`MAE` about `$528,942`), so future modeling should compare against this result while testing nonlinear models, target/outlier handling, and more detailed location features.
